# Notebook 03 — Baseline Recommender

**Purpose:** Implement and evaluate a strong baseline model that will serve as a benchmark and a fallback system for all subsequent models.

**Inputs:**
- `data/splits/train.csv` — historical ratings (train only)
- `data/splits/test.csv` — future ratings (evaluation only)

**Outputs:**
- Rating metrics: RMSE, MAE
- Ranking metrics: Precision@10, NDCG@10
- Popularity ranking table
- Saved model artifact: `results/baseline_model.pkl`

## 0. Imports & Configuration

Set all constants here. No magic numbers in the code below.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import joblib

# Add src/ to path so we can import project modules
sys.path.insert(0, os.path.abspath("../src"))

from baseline import BaselineModel
from evaluation import (
    evaluate_rating_predictions,
    evaluate_ranking,
    run_sanity_checks,
)

# ── Paths ──────────────────────────────────────────────────────
TRAIN_PATH   = "../data/splits/train.csv"
TEST_PATH    = "../data/splits/test.csv"
RESULTS_DIR  = "../results/"

# ── Hyperparameters ───────────────────────────────────────────
POPULARITY_THRESHOLD = 50   # Minimum vote count (m) for Bayesian weighted score
TOP_K                = 10   # Number of recommendations per user
RELEVANCE_THRESHOLD  = 3.5  # Minimum rating to count a test item as "relevant"

# ── Sanity check bounds ───────────────────────────────────────
RMSE_RANGE = (0.8, 1.5)     # Expected RMSE range for a valid baseline
MAE_RANGE  = (0.6, 1.2)     # Expected MAE range for a valid baseline

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Configuration loaded.")

Configuration loaded.


## 1. Load Data

Load train and test sets from disk. This is the ONLY time data is read. Train is used for fitting; test is used only for evaluation at the end.

In [2]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f"Train: {len(train):,} rows | {train['userId'].nunique():,} users | {train['movieId'].nunique():,} movies")
print(f"Test : {len(test):,} rows  | {test['userId'].nunique():,} users | {test['movieId'].nunique():,} movies")
print()
print("Train rating stats:")
print(train["rating"].describe().round(4))

Train: 1,565,182 rows | 12,773 users | 14,246 movies
Test : 397,622 rows  | 12,773 users | 13,902 movies

Train rating stats:
count    1.565182e+06
mean     3.569400e+00
std      1.058200e+00
min      5.000000e-01
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


## 2. Fit the Baseline Model

Call `fit()` with `train` only. Internally this computes:
- μ (global mean)
- b_u per user
- b_i per item
- Popularity ranking via Bayesian weighted score

In [3]:
model = BaselineModel(popularity_threshold=POPULARITY_THRESHOLD)
model.fit(train)

print(f"\nGlobal mean (μ) = {model.mu:.4f}")

[BaselineModel.fit] Done.
  mu              = 3.5694
  unique users    = 12,773
  unique items    = 14,246
  popularity rows = 14,246

Global mean (μ) = 3.5694


## 3. Generate Predictions on Test Set

Vectorised batch prediction. All (userId, movieId) pairs in test receive a prediction. Unknown users or items automatically fall back to the bias hierarchy.

In [4]:
y_pred = model.predict_batch(test)

# Quick spot-check — should have same row count as test, no NaNs
assert len(y_pred) == len(test), f"Prediction count mismatch: {len(y_pred)} vs {len(test)}"
assert not np.isnan(y_pred).any(), "NaN predictions detected — check fallback logic"

print(f"Generated {len(y_pred):,} predictions")
print(f"Prediction range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")
print(f"Prediction mean : {y_pred.mean():.4f}  (expected ≈ {model.mu:.4f})")

Generated 397,622 predictions
Prediction range: [0.50, 5.00]
Prediction mean : 3.5005  (expected ≈ 3.5694)


## 4. Rating Prediction Evaluation (RMSE & MAE)

Evaluate point-wise accuracy. Both metrics compare predicted ratings vs the actual ratings in test.csv.

In [5]:
rating_metrics = evaluate_rating_predictions(test, y_pred)

print("=" * 40)
print("  RATING PREDICTION METRICS")
print("=" * 40)
print(f"  RMSE : {rating_metrics['rmse']:.4f}")
print(f"  MAE  : {rating_metrics['mae']:.4f}")
print(f"  Rows : {rating_metrics['n_preds']:,}")
print(f"  NaN  : {rating_metrics['n_nan_preds']}")

  RATING PREDICTION METRICS
  RMSE : 0.9047
  MAE  : 0.6826
  Rows : 397,622
  NaN  : 0


## 5. Sanity Checks

Validates: no NaN predictions, all values in [0.5, 5.0], RMSE and MAE within expected ranges. ALL checks must pass before we proceed.

In [6]:
checks = run_sanity_checks(
    y_pred,
    rmse=rating_metrics["rmse"],
    mae=rating_metrics["mae"],
    rmse_range=RMSE_RANGE,
    mae_range=MAE_RANGE,
)

print("=" * 40)
print("  SANITY CHECKS")
print("=" * 40)
all_passed = True
for c in checks:
    status = "✅ PASS" if c["passed"] else "❌ FAIL"
    print(f"  {status}  {c['check']}  —  {c['detail']}")
    if not c["passed"]:
        all_passed = False

print()
if all_passed:
    print("  ✅ All checks passed. Baseline is valid.")
else:
    print("  ❌ Some checks FAILED. Investigate before proceeding.")
    raise AssertionError("Baseline sanity checks failed — model is not reliable.")

  SANITY CHECKS
  ✅ PASS  No NaN predictions  —  0 NaN values found
  ✅ PASS  All predictions in [0.5, 5.0]  —  0 out-of-range values
  ✅ PASS  RMSE in (0.8, 1.5)  —  RMSE = 0.9047
  ✅ PASS  MAE in (0.6, 1.2)  —  MAE = 0.6826

  ✅ All checks passed. Baseline is valid.


## 6. Fallback Usage Diagnostics

Report how many test rows triggered the fallback (unknown users or items). This is expected to be low since the temporal split guarantees all test users exist in train.

In [7]:
stats = model.fallback_stats(test)

print("=" * 40)
print("  FALLBACK / COLD-START DIAGNOSTICS")
print("=" * 40)
print(f"  Total test rows    : {stats['total_test_rows']:,}")
print(f"  Unseen users       : {stats['unseen_users']:,}  ({stats['pct_unseen_users']}%)")
print(f"  Unseen items       : {stats['unseen_items']:,}  ({stats['pct_unseen_items']}%)")
print(f"  Both unseen        : {stats['both_unseen']:,}  ({stats['pct_both_unseen']}%)")
print()
print("Leakage check: μ was computed from train ONLY → NO leakage.")

  FALLBACK / COLD-START DIAGNOSTICS
  Total test rows    : 397,622
  Unseen users       : 0  (0.0%)
  Unseen items       : 321  (0.08%)
  Both unseen        : 0  (0.0%)

Leakage check: μ was computed from train ONLY → NO leakage.


## 7. Popularity Ranking (Top-K Recommendations)

Evaluate ranking quality via Precision@10 and NDCG@10. Items the user already rated in train are masked out before ranking. 'Relevant' is defined as rating >= 3.5.

In [8]:
# Define a wrapper that matches the interface expected by evaluate_ranking
def popularity_recommend(userId, k, seen_items):
    top_k_df = model.recommend_top_k(userId, k=k, seen_items=seen_items)
    return top_k_df["movieId"].tolist()

ranking_metrics = evaluate_ranking(
    test=test,
    train=train,
    recommend_fn=popularity_recommend,
    k=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

print("=" * 40)
print("  RANKING METRICS (Popularity Baseline)")
print("=" * 40)
print(f"  Precision@{TOP_K}  : {ranking_metrics['precision_at_k']:.4f}")
print(f"  NDCG@{TOP_K}       : {ranking_metrics['ndcg_at_k']:.4f}")
print(f"  Users evaluated : {ranking_metrics['n_users_evaluated']:,}")
print(f"  Relevance ≥ {RELEVANCE_THRESHOLD}")

  RANKING METRICS (Popularity Baseline)
  Precision@10  : 0.0389
  NDCG@10       : 0.0498
  Users evaluated : 12,623
  Relevance ≥ 3.5


## 8. Top-20 Popular Movies

Inspect the popularity ranking. The weighted score shrinks movies with few votes toward the global mean (Bayesian regularisation).

In [9]:
top20 = model.popularity_df.head(20).copy()
top20.index = range(1, len(top20) + 1)
top20.index.name = "rank"

print(f"Popularity threshold (m) = {POPULARITY_THRESHOLD}")
print(f"Global mean (μ)          = {model.mu:.4f}")
print()
top20[["movieId", "R", "v", "score"]].rename(
    columns={"R": "mean_rating", "v": "n_ratings", "score": "weighted_score"}
).round(4)

Popularity threshold (m) = 50
Global mean (μ)          = 3.5694



,movieId,mean_rating,n_ratings,weighted_score
rank,,,,
1,318,4.4167,6000,4.4097
2,858,4.3225,3712,4.3125
3,50,4.2996,3752,4.2900
4,202439,4.3586,488,4.2853
5,1221,4.2746,2365,4.2600
6,1203,4.2803,1115,4.2498
7,2959,4.2523,4387,4.2446
8,750,4.2592,1732,4.2399
9,5618,4.2508,1812,4.2325


## 9. Final Summary

Collect all results in a single dictionary and save the fitted model to disk for use by subsequent notebooks.

In [10]:
summary = {
    "model":           "BaselineModel (Bias + Popularity)",
    "global_mean_mu":  round(model.mu, 4),
    "n_train_users":   len(model.user_bias),
    "n_train_items":   len(model.item_bias),
    "n_test_rows":     len(test),
    "rmse":            rating_metrics["rmse"],
    "mae":             rating_metrics["mae"],
    f"precision_at_{TOP_K}": ranking_metrics["precision_at_k"],
    f"ndcg_at_{TOP_K}":      ranking_metrics["ndcg_at_k"],
    "unseen_users":     stats["unseen_users"],
    "unseen_items":     stats["unseen_items"],
    "no_leakage":       True,
    "no_nan_preds":     rating_metrics["n_nan_preds"] == 0,
}

print("=" * 50)
print("  FINAL SUMMARY — BASELINE MODEL")
print("=" * 50)
for k, v in summary.items():
    print(f"  {k:<30s} : {v}")

# Save fitted model for downstream notebooks
model_path = os.path.join(RESULTS_DIR, "baseline_model.pkl")
joblib.dump(model, model_path)
print(f"\n✅ Model saved → {model_path}")

  FINAL SUMMARY — BASELINE MODEL
  model                          : BaselineModel (Bias + Popularity)
  global_mean_mu                 : 3.5694
  n_train_users                  : 12773
  n_train_items                  : 14246
  n_test_rows                    : 397622
  rmse                           : 0.90471
  mae                            : 0.682559
  precision_at_10                : 0.038881
  ndcg_at_10                     : 0.049819
  unseen_users                   : 0
  unseen_items                   : 321
  no_leakage                     : True
  no_nan_preds                   : True

✅ Model saved → ../results/baseline_model.pkl
